# Lekcia 03 - Agentové dizajnové vzory

V tejto lekcii preskúmame tri základné dizajnové vzory pre tvorbu efektívnych AI agentov:

1. **Jasné inštrukcie pre agenta** — Tvorba presných, rolou definujúcich výziev, ktoré riadia správanie agenta
2. **Štruktúrovaný výstup s modelmi Pydantic** — Zabezpečenie, aby agenti vracali predvídateľné, overené údaje
3. **Agenti s jedinou zodpovednosťou** — Navrhovanie zameraných agentov, ktorí dobre zvládajú jednu úlohu

Každý vzor aplikujeme na scénar **odporúčania cestovných destinácií**, postupne budujeme systém, ktorý dokáže navrhovať destinácie, kontrolovať dostupnosť a riešiť logistiku.


## Nastavenie


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Vzor 1: Jasné pokyny pre agenta

Najúčinnejší vzor je zároveň najjednoduchší: napísať jasné, podrobné pokyny pre vášho agenta.

Dobré pokyny definujú:
- **Kto** agent je (persona a tón)
- **Čo** má robiť (postupné zodpovednosti)
- **Ako** sa má správať (obmedzenia a štýl)

Nižšie vytvoríme cestovateľského konzultanta s explicitnými pokynmi, ktoré ovplyvňujú každú odpoveď, ktorú vytvorí.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Vzor 2: Štruktúrovaný výstup s modelmi Pydantic

Voľný text je užitočný pre rozhovor, ale následné systémy potrebujú štruktúrované údaje.
Spojením **modelov Pydantic** s **nástrojovou funkciou** môžeme:

- Definovať presné schéma výstupu agenta
- Automaticky overovať odpovede
- Spoľahlivo integrovať výsledky agenta do logiky aplikácie

Tiež zavádzame nástroj, ktorý vracia detaily destinácie, aby agent zakladal svoje odporúčania na skutočných údajoch.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Vzor 3: Agenti s jednou zodpovednosťou

Komplexné úlohy majú prospech zo rozdelenia práce medzi viacerých zameraných agentov, z ktorých každý má jednu zodpovednosť:

- **Expert na destinácie**, ktorý pozná miesta a dostupnosť
- **Plánovač logistiky**, ktorý rieši lety, hotely a itineráre

To odráža princíp softvérového inžinierstva *oddelenie zodpovedností* — každý agent sa ľahšie testuje, udržiava a vylepšuje nezávisle.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Zhrnutie

V tejto lekcii sme aplikovali tri agentné vzory návrhu na scenár odporúčania pre cestovanie:

| Vzor | Hlavná myšlienka | Výhoda |
|---|---|---|
| **Jasné pokyny** | Definovať personu, zodpovednosti a obmedzenia vopred | Konzistentné, značkové správanie agenta |
| **Štruktúrovaný výstup** | Použiť Pydantic modely ako formát odpovede | Overené, strojovo čitateľné výsledky |
| **Jedna zodpovednosť** | Každému agentovi prideliť jednu zameranú úlohu | Jednoduchšie testovanie, údržba a skladanie |

Tieto vzory sa prirodzene kombinujú — môžete spojiť jasné pokyny so štruktúrovaným výstupom v rámci agenta s jednou zodpovednosťou na vytvorenie robustných, produkčne pripravených systémov.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zrieknutie sa zodpovednosti**:  
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Aj keď sa snažíme o presnosť, berte prosím na vedomie, že automatizované preklady môžu obsahovať chyby alebo nepresnosti. Originálny dokument v jeho pôvodnom jazyku by sa mal považovať za autoritatívny zdroj. Pri kritických informáciách sa odporúča profesionálny ľudský preklad. Za akékoľvek nepochopenia alebo nesprávne výklady vyplývajúce z použitia tohto prekladu nenesieme zodpovednosť.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
